In [ ]:
!pip install --upgrade binance-historical-data plotly -q
!pip install requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.8/272.8 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.2 MB/s eta 0:00:00


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from binance_historical_data import BinanceDataDumper
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta, date, time
warnings.filterwarnings('ignore')

In [ ]:
def get_list_all_trading_pairs():
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_list_all_trading_pairs()

def filter_usdt_tickers(tickers):
    exclude_keywords = ["UPUSDT", "DOWNUSDT", "BEARUSDT", "BULLUSDT"]
    return [ticker for ticker in tickers if ticker.endswith("USDT") and not any(ex in ticker for ex in exclude_keywords)]

def find_first_data_date(ticker):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_min_start_date_for_ticker(ticker)

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def detect_timestamp_unit(timestamp):
    num_digits = len(str(timestamp))
    if num_digits == 13:
        return 'ms'
    elif num_digits == 16:
        return 'us'
    else:
        raise ValueError(f"Timestamp không hợp lệ: {timestamp}")

def convert_timestamp(timestamp):
    unit = detect_timestamp_unit(timestamp)
    return pd.to_datetime(timestamp, unit=unit, errors='coerce')

def download_ticker(ticker, date_start, date_end, data_frequency="1h"):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency= data_frequency,
    )
    date_start = datetime.strptime(date_start, "%Y-%m-%d").date()
    date_end = datetime.strptime(date_end, "%Y-%m-%d").date()
    data_dumper.dump_data(
        tickers = ticker,
        date_start = date_start,
        date_end = date_end,
        is_to_update_existing = False,
    )

def read_csv_file(file_path):
    df = pd.read_csv(file_path)
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    df['open_time'] = df['open_time'].apply(convert_timestamp)
    df['close_time'] = df['close_time'].apply(convert_timestamp)
    return df

def get_csv_files(directory):
    try:
        if not os.path.exists(directory):
            print(f"Warning: Thư mục không tồn tại: {directory}")
            return []
        return [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.csv')]
    except Exception as e:
        print(f"Lỗi khi đọc thư mục {directory}: {str(e)}")
        return []

# Load data from file
def process_csv_files(ticker, data_frequency = "1h"):
    daily_path = os.path.join(os.getcwd(), f"spot/daily/klines/{ticker}/{data_frequency}")
    monthly_path = os.path.join(os.getcwd(), f"spot/monthly/klines/{ticker}/{data_frequency}")
    daily_files = get_csv_files(daily_path)
    monthly_files = get_csv_files(monthly_path)
    all_files = daily_files + monthly_files
    if not all_files:
        print(f"❗ Không có file CSV nào cho {ticker}")
        return None
    data = pd.concat([read_csv_file(file) for file in all_files], ignore_index=True)
    data.sort_values(by='open_time', inplace=True)
    return data


In [35]:
data_frequency = "30m"
start_date = "2024-01-01"
end_date = "2025-07-01"
tickers =['BTCUSDT']




In [36]:
download_ticker(tickers[0], start_date, end_date, data_frequency)
crypto_data = process_csv_files(tickers[0], data_frequency)


---> Found overall tickers: 595
---> Filter to asked tickers: 7
------> Tickers left: 2
Download full data for 2 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 30m
---> Start Date: 20240101
---> End Date: 20250701


Tickers:   0%|          | 0/2 [00:00<?, ?it/s]

monthly files to download:   0%|          | 0/18 [00:00<?, ?files/s]

daily files to download:   0%|          | 0/1 [00:00<?, ?files/s]

monthly files to download: 0files [00:00, ?files/s]

daily files to download: 0files [00:00, ?files/s]

Tried to dump data for 2 tickers:
---> For BTCUSDT new data saved for: 18 months 1 days
---> For BTCUSD new data saved for: 0 months 0 days


In [37]:
data = crypto_data[['Close']].reset_index(drop=True)
data.columns = ['Close']
data

,Close
0,42475.23
1,42493.16
2,42613.56
3,42546.01
4,42581.10
...,...
26280,105784.13
26281,105721.22
26282,105613.22
26283,105505.37


In [45]:
import pandas as pd
import numpy as np

def backtest(data, initial_balance=100, position_size=0.002, stop_loss=-2, take_profit=5):
    # Calculate SMAs and SMA_distance
    data['SMA_200'] = data['Close'].rolling(window=200).mean()
    data['SMA_50'] = data['Close'].rolling(window=50).mean()
    data['SMA_10'] = data['Close'].rolling(window=10).mean()
    data['SMA_distance'] = abs(data['Close'] - data['SMA_200'])
    distance = np.array(data['SMA_distance'].dropna())

    # Initialize backtest variables
    balance = initial_balance
    position = None  # {'type': 'buy'/'sell', 'entry_price': float, 'size': float}
    trades = []  # To store trade history
    quantile_threshold = 0.6
    # Iterate through the data
    for i in range(3, len(data)):
        # Check if no position is open
        if position is None:
            # Buy condition
            if (data['SMA_distance'].iloc[i-1] < np.quantile(distance, quantile_threshold) and
                data['Close'].iloc[i-1] < data['SMA_200'].iloc[i-1]):
                if (data['SMA_10'].iloc[i-1] > data['SMA_50'].iloc[i-1] and
                    data['SMA_10'].iloc[i-2] < data['SMA_50'].iloc[i-2]):
                    position = {
                        'type': 'buy',
                        'entry_price': data['Close'].iloc[i],
                        'size': position_size
                    }
                    trades.append({
                        'type': 'buy',
                        'entry_index': i,
                        'entry_price': data['Close'].iloc[i],
                        'entry_time': data.index[i]
                    })

            # Sell condition
            elif (data['SMA_distance'].iloc[i-1] < np.quantile(distance, quantile_threshold) and
                  data['Close'].iloc[i-1] > data['SMA_200'].iloc[i-1]):
                if (data['SMA_10'].iloc[i-1] < data['SMA_50'].iloc[i-1] and
                    data['SMA_10'].iloc[i-2] > data['SMA_50'].iloc[i-2]):
                    position = {
                        'type': 'sell',
                        'entry_price': data['Close'].iloc[i],
                        'size': position_size
                    }
                    trades.append({
                        'type': 'sell',
                        'entry_index': i,
                        'entry_price': data['Close'].iloc[i],
                        'entry_time': data.index[i]
                    })

        # Check if position is open
        elif position is not None:
            current_price = data['Close'].iloc[i]
            if position['type'] == 'buy':
                unrealized_profit = (current_price - position['entry_price']) * position['size']
            else:  # sell
                unrealized_profit = (position['entry_price'] - current_price) * position['size']

            # Close position based on stop-loss or take-profit
            if unrealized_profit < stop_loss or unrealized_profit > take_profit:
                trades[-1]['exit_index'] = i
                trades[-1]['exit_price'] = current_price
                trades[-1]['exit_time'] = data.index[i]
                trades[-1]['profit'] = unrealized_profit
                balance += unrealized_profit
                position = None

    # If a position is still open at the end, close it at the last price
    if position is not None:
        current_price = data['Close'].iloc[-1]
        if position['type'] == 'buy':
            unrealized_profit = (current_price - position['entry_price']) * position['size']
        else:  # sell
            unrealized_profit = (position['entry_price'] - current_price) * position['size']
        trades[-1]['exit_index'] = len(data) - 1
        trades[-1]['exit_price'] = current_price
        trades[-1]['exit_time'] = data.index[-1]
        trades[-1]['profit'] = unrealized_profit
        balance += unrealized_profit

    # Create trades DataFrame
    trades_df = pd.DataFrame(trades)

    return {
        'final_balance': balance,
        'trades': trades_df,
        'initial_balance': initial_balance
    }

In [46]:
result = backtest(data)
print(f"Final Balance: {result['final_balance']}")
print(result['trades'])

Final Balance: 111.2651599999999
     type  entry_index  entry_price  entry_time  exit_index  exit_price  \
0     buy          209     43975.46         209         372    46970.01   
1    sell          426     45439.44         426         468    46527.33   
2    sell          519     46784.29         519         559    44122.60   
3     buy          689     42685.62         689         851    41470.59   
4     buy          904     41620.00         904        1027    40386.79   
..    ...          ...          ...         ...         ...         ...   
147   buy        25663    104944.06       25663       25739   103940.01   
148   buy        25784    103866.97       25784       25795   102645.22   
149  sell        26063    106998.53       26063       26162   108138.55   
150  sell        26183    107359.79       26183       26188   108388.99   
151  sell        26208    107650.00       26208       26284   105681.14   

     exit_time   profit  
0          372  5.98910  
1          468